# Lead-lag and the vintage test

Sign convention: a positive lag k pairs the signal at month t with the outcome at month t + k, so the signal leads by k. Predictive regressions use OLS with Newey-West standard errors (maxlags equal to the horizon). The inference describes the fitted relationship; M3 is a voluntary non-probability panel, and no sampling interval attaches to its levels.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from factory_cycle.loaders import load_m3_levels, load_g17, load_alfred_vintages
from factory_cycle import signals as sig
pd.set_option("display.width", 140)
m3 = load_m3_levels("../data/staged/m3_levels.parquet")
g17 = load_g17("../data/staged/g17.parquet")
panel = sig.build_signals_panel(m3)
from factory_cycle import leadlag, vintages
outcomes = leadlag.forward_outcomes(g17)
mtm = panel[panel['aggregate'] == 'MTM'].set_index('month')


## Backlog months: cross-correlations and regressions by window

In [2]:
ev = leadlag.evaluate_signal(mtm, outcomes, 'backlog_months')
rows = []
for outcome in ['mcumfn', 'ipman']:
    for window, entry in ev[outcome].items():
        reg = entry['reg_12m']
        rows.append({'outcome': outcome, 'window': window,
                     'best_lag': entry['xcorr']['best_lag'],
                     'best_corr': entry['xcorr']['best_corr'],
                     'coef_12m': reg['coef'], 't_12m': reg['t_stat'],
                     'r2': reg['r2'], 'n': reg['n']})
pd.DataFrame(rows)

,outcome,window,best_lag,best_corr,coef_12m,t_12m,r2,n
0,mcumfn,full,0,-0.428637,0.694711,0.725615,0.007273,405
1,mcumfn,1992_2007,-12,0.146786,-2.306981,-0.565381,0.018269,192
2,mcumfn,2008_2019,-12,0.561749,1.853207,0.544199,0.010874,144
3,mcumfn,2020_present,0,-0.833821,16.852213,3.483046,0.586283,69
4,mcumfn,excl_2020_2021,3,-0.412389,0.149853,0.161478,0.000364,381
5,ipman,full,0,-0.512641,-3.620280,-2.791353,0.091209,405
6,ipman,1992_2007,12,-0.249076,-6.194406,-1.028686,0.062039,192
7,ipman,2008_2019,1,-0.401120,1.216986,0.262384,0.002633,144
8,ipman,2020_present,12,0.803494,23.906487,3.823322,0.645602,69
9,ipman,excl_2020_2021,0,-0.526017,-4.369731,-3.446878,0.137536,381


## Stability, stated plainly

The helper flags every sign change and every significance flip in the 12-month regressions across windows.

In [3]:
leadlag.stability_note(ev)

['capacity utilization: the 12-month coefficient changes sign across windows (full span +0.69, 1992 to 2007 -2.31, 2008 to 2019 +1.85, 2020 to present +16.85, excluding 2020 to 2021 +0.15); the relationship is not stable.',
 'capacity utilization: 12-month significance at 5% flips across windows (full span not significant, 1992 to 2007 not significant, 2008 to 2019 not significant, 2020 to present significant, excluding 2020 to 2021 not significant).',
 'IP growth: the 12-month coefficient changes sign across windows (full span -3.62, 1992 to 2007 -6.19, 2008 to 2019 +1.22, 2020 to present +23.91, excluding 2020 to 2021 -4.37); the relationship is not stable.',
 'IP growth: 12-month significance at 5% flips across windows (full span significant, 1992 to 2007 not significant, 2008 to 2019 not significant, 2020 to present significant, excluding 2020 to 2021 significant).']

## Vintage matrices from the ALFRED intervals

Each cell is the value whose effective interval contains the column's vintage date. The builder can only select an interval that had already started at that date.

In [4]:
alfred = load_alfred_vintages('../data/staged/alfred_vintages_long.parquet')
uo = vintages.build_vintage_matrix(alfred, 'AMTMUO')
vs = vintages.build_vintage_matrix(alfred, 'AMTMVS')
print('matrix shape (obs months x vintages):', uo.shape)
uo.iloc[-3:, -3:]

matrix shape (obs months x vintages): (417, 191)


vintage_date,2026-09-02,2026-10-02,2026-11-03
obs_month,,,
2026-07-01,1600284.0,1599680.0,1599680.0
2026-08-01,NaN,1609602.0,1610037.0
2026-09-01,NaN,NaN,1623818.0


## First published versus final backlog months

In [5]:
first_bl = (vintages.first_release(uo)['first_value'] /
            vintages.first_release(vs)['first_value'])
final_bl = vintages.final_values(uo) / vintages.final_values(vs)
diff = (first_bl - final_bl).abs().dropna()
{'n_months': int(len(diff)),
 'mean_abs_diff_months': float(diff.mean()),
 'max_abs_diff_months': float(diff.max()),
 'share_gt_0_05': float((diff > 0.05).mean())}

{'n_months': 417,
 'mean_abs_diff_months': 0.06344723322134749,
 'max_abs_diff_months': 0.2705296908814159,
 'share_gt_0_05': 0.4412470023980815}

## The AMTMUO revision triangle (first rows)

Horizon columns hold the value in effect 3, 6, and 12 months after the first release date; they are empty when that date falls after the last vintage in the record.

In [6]:
tri = vintages.revision_triangle(alfred, 'AMTMUO', obs_start='2011-01-01')
tri.head(6)

,obs_month,first_release,first_vintage,plus_3m,plus_6m,plus_12m,latest
0,2011-01-01,838186.0,2011-06-24,838186.0,838186.0,887116.0,879376.0
1,2011-02-01,842152.0,2011-06-24,842152.0,842152.0,888337.0,881311.0
2,2011-03-01,848202.0,2011-06-24,848202.0,848202.0,897489.0,891823.0
3,2011-04-01,850721.0,2011-06-24,853164.0,853164.0,901244.0,894996.0
4,2011-05-01,860907.0,2011-07-05,860748.0,860748.0,909446.0,903737.0
5,2011-06-01,863237.0,2011-08-03,863541.0,863541.0,913890.0,905689.0


In [7]:
vintages.triangle_revision_stats(tri)

{'n': 189,
 'mean_revision_pct': 2.127137326745143,
 'mean_abs_revision_pct': 3.8250936357196137,
 'median_abs_revision_pct': 3.4696221957154108}